In [1]:
import os
import re
import json
import itertools
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
font1 = {"weight": "bold", "size": 18}

## Specify rts-gmlc path and other data path

In [7]:
gmlc_path = os.path.join(os.getcwd(), "..", "..", "rts-gmlc", "RTS-GMLC", "RTS_Data", "timeseries_data_files")
root_path = os.path.join(os.getcwd(), "..", "Data")

## Read Bus Renewable Generation

In [8]:
gen_csv_path = os.path.join(gmlc_path, "..", "SourceData", "gen.csv")
bus_csv_path = os.path.join(gmlc_path, "..", "SourceData", "bus.csv")
df_gen = pd.read_csv(gen_csv_path, index_col=False)
df_bus = pd.read_csv(bus_csv_path, index_col=False)
# make a dictionary, keys are bus ids
bus_renew_gen = {}
for b in df_bus['Bus ID']:
    bus_renew_gen[str(b)] = np.zeros(366*24)

# for every renew generator, find which bus it locates, and store the renew generation in the dictionary
# hydro
hydro_da_path = os.path.join(gmlc_path, "Hydro", "DAY_AHEAD_hydro.csv")
df_da_hydro = pd.read_csv(hydro_da_path)
hydro_gen_names = df_da_hydro.columns[4:]

for name in hydro_gen_names:
    bus_id = name.split("_")[0]
    bus_renew_gen[bus_id] += df_da_hydro[name].to_numpy()

# csp
csp_da_path = os.path.join(gmlc_path, "CSP", "DAY_AHEAD_Natural_Inflow.csv")
df_da_csp = pd.read_csv(csp_da_path)
csp_da_arr = df_da_csp["212_CSP_1"].to_numpy()

bus_renew_gen["212"] += csp_da_arr

# pv
pv_da_path = os.path.join(gmlc_path, "PV", "DAY_AHEAD_pv.csv")
df_da_pv = pd.read_csv(pv_da_path)
pv_gen_names = df_da_pv.columns[4:]

for name in pv_gen_names:
    bus_id = name.split("_")[0]
    bus_renew_gen[bus_id] += df_da_pv[name].to_numpy()

# RTPV
rtpv_da_path = os.path.join(gmlc_path, "RTPV", "DAY_AHEAD_rtpv.csv")
df_da_rtpv = pd.read_csv(rtpv_da_path)
rtpv_gen_names = df_da_rtpv.columns[4:]

for name in rtpv_gen_names:
    bus_id = name.split("_")[0]
    bus_renew_gen[bus_id] += df_da_rtpv[name].to_numpy()

# wind
wind_da_path = os.path.join(gmlc_path, "WIND", "DAY_AHEAD_wind.csv")
df_da_wind = pd.read_csv(wind_da_path)
wind_gen_names = df_da_wind.columns[4:]

for name in wind_gen_names:
    bus_id = name.split("_")[0]
    bus_renew_gen[bus_id] += df_da_wind[name].to_numpy()

# pd.DataFrame(bus_renew_gen).to_csv("bus_renewable.csv")

## Read Bus Load

In [12]:
df_buses = pd.read_csv(os.path.join(root_path, "base_pcm_simulation_new_env", "bus_detail.csv"))

# loop over all buses
df_all_bus = pd.read_csv(bus_csv_path)
bus_names = df_all_bus['Bus Name'].to_list()
bus_ids = df_all_bus['Bus ID'].to_list()
bus_load_dict = {}

for idx, name in enumerate(bus_names):
    bus_load = df_buses[df_buses["Bus"]==name]["Demand"].to_numpy()
    bus_id = bus_ids[idx]
    bus_load_dict[bus_id] = bus_load

df = pd.DataFrame(bus_load_dict)

# df

## Read Bus Dispatchable (fossil or nuclear) Capacity 

In [35]:
df_gen["Unit Type"] == "CC"

0      False
1      False
2      False
3      False
4      False
       ...  
153    False
154    False
155    False
156    False
157    False
Name: Unit Type, Length: 158, dtype: bool

In [47]:
# max capacity

bus_fossil_max_capacity_dict = {}
bus_ids = df_all_bus['Bus ID'].to_list()

for id in bus_ids:
    bus_fossil_max_capacity_dict[id] = 0

for index, row in df_gen.iterrows():
    if (row["Unit Type"] == "CC") | (row["Unit Type"] == "CT") | (row["Unit Type"] == "STEAM") | (row["Unit Type"] == "NUCLEAR"):
        bus_fossil_max_capacity_dict[row["Bus ID"]] += row["PMax MW"]

# bus_fossil_max_capacity_dict

In [48]:
# min capacity

bus_fossil_min_capacity_dict = {}
bus_ids = df_all_bus['Bus ID'].to_list()

for id in bus_ids:
    bus_fossil_min_capacity_dict[id] = 0

for index, row in df_gen.iterrows():
    if (row["Unit Type"] == "CC") | (row["Unit Type"] == "CT") | (row["Unit Type"] == "STEAM") | (row["Unit Type"] == "NUCLEAR"):
        bus_fossil_min_capacity_dict[row["Bus ID"]] += row["PMax MW"]

# bus_fossil_min_capacity_dict

In [76]:
# Vmag
bus_vmag_dict = {}

for idx, row in df_all_bus.iterrows():
    mag = row["V Mag"]
    id = row["Bus ID"]
    bus_vmag_dict[id] = mag
    
# bus_vmag_dict

In [77]:
# V angle
bus_vangle_dict = {}

for idx, row in df_all_bus.iterrows():
    ang = row["V Angle"]
    id = row["Bus ID"]
    bus_vangle_dict[id] = ang

# bus_vangle_dict

In [78]:
# Bus Type, PQ = 0 and PV = 1
bus_type_dict = {}

for idx, row in df_all_bus.iterrows():
    tp = row["Bus Type"]
    id = row["Bus ID"]
    if tp == "PV":
        bus_type_dict[id] = 1
    else:
        bus_type_dict[id] = 0

# bus_type_dict

## Static features

In [81]:
bus_ids = df_bus["Bus ID"].to_list()

static_feature_dict = {}
for id in bus_ids:
    static_feature_dict[id] = []
    # append max dispatchable capacity
    static_feature_dict[id].append(bus_fossil_max_capacity_dict[id])
    # append min dispatchable capacity
    static_feature_dict[id].append(bus_fossil_min_capacity_dict[id])    
    # append V mag
    static_feature_dict[id].append(bus_vmag_dict[id])
    # append V angle
    static_feature_dict[id].append(bus_vangle_dict[id])
    # append bus type
    static_feature_dict[id].append(bus_type_dict[id])

df_static = pd.DataFrame(static_feature_dict)

df_static.to_csv("bus_static.csv")

## Extract LMP

In [90]:
df_buses = pd.read_csv(os.path.join(root_path, "base_pcm_simulation_new_env", "bus_detail.csv"))

# make a map from id to name
id_name_map = {}
for idx, row in df_bus.iterrows():
    id_name_map[row["Bus Name"]] = row["Bus ID"]

lmp_by_bus = {}
for name in id_name_map.keys():
    id = id_name_map[name]
    lmp_by_bus[id] = df_buses[df_buses["Bus"]==name]["LMP DA"].to_numpy()

pd.DataFrame(lmp_by_bus).to_csv("lmp_by_bus.csv")